# DeepSeek ChildSnack Reasoning Tokens

Count the tokens in every non-review `reasoning.txt` produced from the updated ChildSnack problem descriptions. The notebook averages token counts within each ten-problem batch and plots regular-formalizer, higher-order-formalizer, and planner curves against the number of children.

The tokenizer is downloaded and cached at runtime. `deepseek-ai/DeepSeek-V3` is the default official published DeepSeek tokenizer; set `DEEPSEEK_TOKENIZER_ID` before running if a tokenizer specifically published for `deepseek-v4-flash` is available.


In [ ]:
# Cell 1: Install missing runtime dependencies into the active Jupyter kernel
import importlib.util
import subprocess
import sys

required_packages = {
    "matplotlib": "matplotlib",
    "transformers": "transformers",
}
missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing_packages]
    )
else:
    print("Required packages are already installed in this kernel.")


In [ ]:
# Cell 2: Imports, project paths, tokenizer configuration, and batch definitions
import os
from pathlib import Path
from statistics import fmean

import matplotlib.pyplot as plt
from transformers import AutoTokenizer


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "output").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the formalizer-scaling project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
TOKENIZER_ID = os.getenv("DEEPSEEK_TOKENIZER_ID", "deepseek-ai/DeepSeek-V3")
TOKENIZER_CACHE = PROJECT_ROOT / ".cache" / "huggingface"

METHOD_ROOTS = {
    "Regular Formalizer": (
        PROJECT_ROOT
        / "output"
        / "direct_pddl_direct_unraveling"
        / "ChildSnack"
        / "single_sandwich_line"
        / "updated_deepseek_v4_flash"
    ),
    "Higher-Order Formalizer": (
        PROJECT_ROOT
        / "output"
        / "direct_pddl_unraveling"
        / "ChildSnack"
        / "single_sandwich_line"
        / "updated_deepseek_v4_flash"
    ),
    "Planner": (
        PROJECT_ROOT
        / "output"
        / "llm_planner_direct_unraveling"
        / "ChildSnack"
        / "single_sandwich_line"
        / "updated_deepseek_v4_flash"
    ),
}

BATCHES = [
    {"start": 1, "end": 10, "children": 5},
    {"start": 11, "end": 20, "children": 10},
    {"start": 21, "end": 30, "children": 15},
    {"start": 31, "end": 40, "children": 20},
    {"start": 41, "end": 50, "children": 25},
    {"start": 51, "end": 60, "children": 30},
    {"start": 61, "end": 70, "children": 50},
    {"start": 71, "end": 80, "children": 100},
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TOKENIZER_ID:", TOKENIZER_ID)
for method, root in METHOD_ROOTS.items():
    print(f"{method}: {root}")


In [ ]:
# Cell 3: Load the DeepSeek tokenizer at runtime
TOKENIZER_CACHE.mkdir(parents=True, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    cache_dir=str(TOKENIZER_CACHE),
    use_fast=True,
)

print("Loaded tokenizer:", TOKENIZER_ID)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Cache directory:", TOKENIZER_CACHE)


In [ ]:
# Cell 4: Locate and validate all non-review reasoning traces
def problem_id(number: int) -> str:
    return f"p{number:02d}"


trace_records = []
missing_files = []
empty_files = []

for method, method_root in METHOD_ROOTS.items():
    for batch in BATCHES:
        batch_label = f"p{batch['start']:02d}-p{batch['end']:02d}"
        for number in range(batch["start"], batch["end"] + 1):
            pid = problem_id(number)
            path = method_root / pid / "cand_00" / "reasoning.txt"
            if not path.is_file():
                missing_files.append(path)
                continue

            text = path.read_text(encoding="utf-8")
            if not text.strip():
                empty_files.append(path)
                continue

            trace_records.append(
                {
                    "method": method,
                    "batch": batch_label,
                    "children": batch["children"],
                    "problem_id": pid,
                    "path": path,
                    "text": text,
                }
            )

if missing_files or empty_files:
    details = [
        *(f"Missing: {path}" for path in missing_files),
        *(f"Empty: {path}" for path in empty_files),
    ]
    raise FileNotFoundError(
        "Reasoning-trace coverage is incomplete; refusing to calculate biased averages.\n"
        + "\n".join(details)
    )

expected_count = len(METHOD_ROOTS) * sum(
    batch["end"] - batch["start"] + 1 for batch in BATCHES
)
assert len(trace_records) == expected_count
print(
    f"Validated {len(trace_records)} reasoning traces "
    f"({expected_count // len(METHOD_ROOTS)} per method)."
)


In [ ]:
# Cell 5: Count tokens in each reasoning trace
def count_reasoning_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


for record in trace_records:
    record["reasoning_tokens"] = count_reasoning_tokens(record["text"])

print("Per-problem token counts:")
print(f"{'Method':<28} {'Problem':<8} {'Children':>8} {'Tokens':>10}")
print("-" * 58)
for record in trace_records:
    print(
        f"{record['method']:<28} "
        f"{record['problem_id']:<8} "
        f"{record['children']:>8} "
        f"{record['reasoning_tokens']:>10,}"
    )


In [ ]:
# Cell 6: Calculate average reasoning tokens for each method and batch
batch_averages = []

for batch in BATCHES:
    batch_label = f"p{batch['start']:02d}-p{batch['end']:02d}"
    for method in METHOD_ROOTS:
        counts = [
            record["reasoning_tokens"]
            for record in trace_records
            if record["method"] == method and record["batch"] == batch_label
        ]
        expected_batch_size = batch["end"] - batch["start"] + 1
        if len(counts) != expected_batch_size:
            raise ValueError(
                f"Expected {expected_batch_size} traces for {method}, {batch_label}; "
                f"found {len(counts)}."
            )

        batch_averages.append(
            {
                "method": method,
                "batch": batch_label,
                "children": batch["children"],
                "problem_count": len(counts),
                "average_reasoning_tokens": fmean(counts),
            }
        )

print("Average reasoning tokens per batch:")
print(f"{'Method':<28} {'Batch':<11} {'Children':>8} {'Problems':>9} {'Average tokens':>16}")
print("-" * 78)
for row in batch_averages:
    print(
        f"{row['method']:<28} "
        f"{row['batch']:<11} "
        f"{row['children']:>8} "
        f"{row['problem_count']:>9} "
        f"{row['average_reasoning_tokens']:>16,.2f}"
    )


In [ ]:
# Cell 7: Plot average reasoning tokens against the main scaling variable
TITLE_FONTSIZE = 24
AXIS_LABEL_FONTSIZE = 20
TICK_LABEL_FONTSIZE = 17
LEGEND_FONTSIZE = 18
ANNOTATION_FONTSIZE = 15

from matplotlib.font_manager import FontProperties
from matplotlib.transforms import Bbox

styles = {
    "Regular Formalizer": {
        "color": "#0B6E75",
        "marker": "o",
        "label_positions": [
            (0, 18, "center", "bottom"),
            (-20, 14, "right", "bottom"),
            (20, 14, "left", "bottom"),
            (0, -20, "center", "top"),
        ],
    },
    "Higher-Order Formalizer": {
        "color": "#D97706",
        "marker": "s",
        "label_positions": [
            (0, -20, "center", "top"),
            (-20, -16, "right", "top"),
            (20, -16, "left", "top"),
            (0, 18, "center", "bottom"),
        ],
    },
    "Planner": {
        "color": "#A63D40",
        "marker": "^",
        "label_positions": [
            (16, 14, "left", "bottom"),
            (0, 18, "center", "bottom"),
            (-16, 14, "right", "bottom"),
            (16, -16, "left", "top"),
            (0, -20, "center", "top"),
        ],
    },
}

LABEL_FALLBACK_POSITIONS = [
    (-22, -18, "right", "top"),
    (22, -18, "left", "top"),
    (0, 40, "center", "bottom"),
    (0, -42, "center", "top"),
    (38, 0, "left", "center"),
    (-38, 0, "right", "center"),
    (34, 28, "left", "bottom"),
    (-34, 28, "right", "bottom"),
    (34, -28, "left", "top"),
    (-34, -28, "right", "top"),
]
LABEL_FONT = FontProperties(size=ANNOTATION_FONTSIZE, weight="bold")


def _overlap_area(first, second):
    width = max(0, min(first.x1, second.x1) - max(first.x0, second.x0))
    height = max(0, min(first.y1, second.y1) - max(first.y0, second.y0))
    return width * height


def _label_bbox(anchor, width, height, horizontal_alignment, vertical_alignment, padding=4):
    anchor_x, anchor_y = anchor
    if horizontal_alignment == "center":
        left = anchor_x - width / 2
    elif horizontal_alignment == "right":
        left = anchor_x - width
    else:
        left = anchor_x

    if vertical_alignment == "center":
        bottom = anchor_y - height / 2
    elif vertical_alignment == "top":
        bottom = anchor_y - height
    else:
        bottom = anchor_y

    return Bbox.from_extents(
        left - padding,
        bottom - padding,
        left + width + padding,
        bottom + height + padding,
    )


def annotate_non_overlapping(ax, points, obstacle_artists=()):
    """Place crowded value labels first and choose the first collision-free position."""
    figure = ax.figure
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    axes_bbox = ax.get_window_extent(renderer=renderer)
    points_per_inch = figure.dpi / 72

    display_points = [ax.transData.transform((point["x"], point["y"])) for point in points]
    marker_boxes = [
        Bbox.from_extents(x - 8, y - 8, x + 8, y + 8)
        for x, y in display_points
    ]

    occupied = []
    for artist in obstacle_artists:
        if artist is not None:
            obstacle = artist.get_window_extent(renderer=renderer)
            occupied.append(
                Bbox.from_extents(
                    obstacle.x0 - 6, obstacle.y0 - 6, obstacle.x1 + 6, obstacle.y1 + 6
                )
            )

    crowded_first = []
    for index, display_point in enumerate(display_points):
        distances = [
            ((display_point[0] - other[0]) ** 2 + (display_point[1] - other[1]) ** 2) ** 0.5
            for other_index, other in enumerate(display_points)
            if other_index != index
        ]
        crowded_first.append((min(distances, default=float("inf")), index))

    for _, point_index in sorted(crowded_first):
        point = points[point_index]
        point_display = display_points[point_index]
        label = f"{point['y']:,.0f}"
        width, height, descent = renderer.get_text_width_height_descent(
            label, LABEL_FONT, ismath=False
        )
        height += descent

        preferred = point["style"]["label_positions"]
        candidates = preferred + [
            position for position in LABEL_FALLBACK_POSITIONS if position not in preferred
        ]
        best = None

        for priority, (offset_x, offset_y, horizontal_alignment, vertical_alignment) in enumerate(candidates):
            anchor = (
                point_display[0] + offset_x * points_per_inch,
                point_display[1] + offset_y * points_per_inch,
            )
            candidate_bbox = _label_bbox(
                anchor, width, height, horizontal_alignment, vertical_alignment
            )

            inside_width = max(
                0, min(candidate_bbox.x1, axes_bbox.x1) - max(candidate_bbox.x0, axes_bbox.x0)
            )
            inside_height = max(
                0, min(candidate_bbox.y1, axes_bbox.y1) - max(candidate_bbox.y0, axes_bbox.y0)
            )
            outside_area = candidate_bbox.width * candidate_bbox.height - inside_width * inside_height
            label_overlap = sum(_overlap_area(candidate_bbox, box) for box in occupied)
            marker_overlap = sum(
                _overlap_area(candidate_bbox, box)
                for marker_index, box in enumerate(marker_boxes)
                if marker_index != point_index
            )
            score = outside_area * 1_000_000 + label_overlap * 1_000 + marker_overlap * 10 + priority

            if best is None or score < best[0]:
                best = (
                    score, offset_x, offset_y, horizontal_alignment, vertical_alignment, candidate_bbox
                )
            if outside_area == 0 and label_overlap == 0 and marker_overlap == 0:
                break

        _, offset_x, offset_y, horizontal_alignment, vertical_alignment, label_bbox = best
        ax.annotate(
            label,
            xy=(point["x"], point["y"]),
            xytext=(offset_x, offset_y),
            textcoords="offset points",
            color=point["style"]["color"],
            fontsize=ANNOTATION_FONTSIZE,
            fontweight="bold",
            ha=horizontal_alignment,
            va=vertical_alignment,
        )
        occupied.append(label_bbox)

fig, ax = plt.subplots(figsize=(12.5, 7.5))
fig.patch.set_facecolor("#F6F1E8")
ax.set_facecolor("#FFFDF8")
childsnack_label_points = []

for method, style in styles.items():
    rows = sorted(
        (row for row in batch_averages if row["method"] == method),
        key=lambda row: row["children"],
    )
    ax.plot(
        [row["children"] for row in rows],
        [row["average_reasoning_tokens"] for row in rows],
        label=method,
        color=style["color"],
        marker=style["marker"],
        linewidth=3.2,
        markersize=10,
    )

    childsnack_label_points.extend(
        {"x": row["children"], "y": row["average_reasoning_tokens"], "style": style}
        for row in rows
    )

child_counts = [batch["children"] for batch in BATCHES]
ax.set_xticks(child_counts)
ax.set_xlabel("Number of children", fontsize=AXIS_LABEL_FONTSIZE, fontweight="semibold", labelpad=12)
ax.set_ylabel("Average reasoning tokens per problem", fontsize=AXIS_LABEL_FONTSIZE, fontweight="semibold", labelpad=12)
ax.set_title(
    "DeepSeek ChildSnack Reasoning Tokens",
    fontsize=TITLE_FONTSIZE,
    fontweight="bold",
    pad=18,
)
ax.grid(axis="y", color="#D8D1C4", linewidth=0.8, alpha=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
childsnack_legend = ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
ax.ticklabel_format(axis="y", style="plain")
ax.margins(y=0.14)
fig.tight_layout()
annotate_non_overlapping(ax, childsnack_label_points, obstacle_artists=[childsnack_legend])
plt.show()


## Elevators

The Elevators results use the same three DeepSeek methods and token metric as ChildSnack, but are aggregated and plotted in a separate figure.

In [ ]:
# Cell 8: Elevators result paths and floor-count batches
ELEVATORS_METHOD_ROOTS = {
    "Regular Formalizer": (
        PROJECT_ROOT
        / "output"
        / "direct_pddl_direct_unraveling"
        / "Elevators"
        / "bottom_to_top_delivery"
        / "deepseek_v4_flash"
    ),
    "Higher-Order Formalizer": (
        PROJECT_ROOT
        / "output"
        / "direct_pddl_unraveling"
        / "Elevators"
        / "bottom_to_top_delivery"
        / "deepseek_v4_flash"
    ),
    "Planner": (
        PROJECT_ROOT
        / "output"
        / "llm_planner_direct_unraveling"
        / "Elevators"
        / "bottom_to_top_delivery"
        / "deepseek_v4_flash"
    ),
}

ELEVATORS_BATCHES = [
    {"start": 1, "end": 10, "floors": 10},
    {"start": 11, "end": 20, "floors": 15},
    {"start": 21, "end": 30, "floors": 20},
    {"start": 31, "end": 40, "floors": 25},
    {"start": 41, "end": 50, "floors": 30},
    {"start": 51, "end": 60, "floors": 50},
    {"start": 61, "end": 70, "floors": 100},
]

for method, root in ELEVATORS_METHOD_ROOTS.items():
    print(f"{method}: {root}")

In [ ]:
# Cell 9: Locate and validate all Elevators reasoning traces
elevators_trace_records = []
elevators_missing_files = []
elevators_empty_files = []

for method, method_root in ELEVATORS_METHOD_ROOTS.items():
    for batch in ELEVATORS_BATCHES:
        batch_label = f"p{batch['start']:02d}-p{batch['end']:02d}"
        for number in range(batch["start"], batch["end"] + 1):
            pid = problem_id(number)
            path = method_root / pid / "cand_00" / "reasoning.txt"
            if not path.is_file():
                elevators_missing_files.append(path)
                continue

            text = path.read_text(encoding="utf-8")
            if not text.strip():
                elevators_empty_files.append(path)
                continue

            elevators_trace_records.append(
                {
                    "method": method,
                    "batch": batch_label,
                    "floors": batch["floors"],
                    "problem_id": pid,
                    "path": path,
                    "text": text,
                }
            )

if elevators_missing_files or elevators_empty_files:
    details = [
        *(f"Missing: {path}" for path in elevators_missing_files),
        *(f"Empty: {path}" for path in elevators_empty_files),
    ]
    raise FileNotFoundError(
        "Elevators reasoning-trace coverage is incomplete; refusing to calculate biased averages.\n"
        + "\n".join(details)
    )

expected_elevators_count = len(ELEVATORS_METHOD_ROOTS) * sum(
    batch["end"] - batch["start"] + 1 for batch in ELEVATORS_BATCHES
)
assert len(elevators_trace_records) == expected_elevators_count
print(
    f"Validated {len(elevators_trace_records)} Elevators reasoning traces "
    f"({expected_elevators_count // len(ELEVATORS_METHOD_ROOTS)} per method)."
)

In [ ]:
# Cell 10: Count and average Elevators reasoning tokens
for record in elevators_trace_records:
    record["reasoning_tokens"] = count_reasoning_tokens(record["text"])


elevators_batch_averages = []
for batch in ELEVATORS_BATCHES:
    batch_label = f"p{batch['start']:02d}-p{batch['end']:02d}"
    for method in ELEVATORS_METHOD_ROOTS:
        counts = [
            record["reasoning_tokens"]
            for record in elevators_trace_records
            if record["method"] == method and record["batch"] == batch_label
        ]
        expected_batch_size = batch["end"] - batch["start"] + 1
        if len(counts) != expected_batch_size:
            raise ValueError(
                f"Expected {expected_batch_size} traces for {method}, {batch_label}; "
                f"found {len(counts)}."
            )

        elevators_batch_averages.append(
            {
                "method": method,
                "batch": batch_label,
                "floors": batch["floors"],
                "problem_count": len(counts),
                "average_reasoning_tokens": fmean(counts),
            }
        )

print("Elevators average reasoning tokens per batch:")
print(f"{'Method':<28} {'Batch':<11} {'Floors':>8} {'Problems':>9} {'Average tokens':>16}")
print("-" * 78)
for row in elevators_batch_averages:
    print(
        f"{row['method']:<28} "
        f"{row['batch']:<11} "
        f"{row['floors']:>8} "
        f"{row['problem_count']:>9} "
        f"{row['average_reasoning_tokens']:>16,.2f}"
    )

In [ ]:
# Cell 11: Plot Elevators reasoning tokens in a separate figure
elevators_fig, elevators_ax = plt.subplots(figsize=(12.5, 7.5))
elevators_fig.patch.set_facecolor("#F6F1E8")
elevators_ax.set_facecolor("#FFFDF8")
elevators_label_points = []

for method, style in styles.items():
    rows = sorted(
        (row for row in elevators_batch_averages if row["method"] == method),
        key=lambda row: row["floors"],
    )
    elevators_ax.plot(
        [row["floors"] for row in rows],
        [row["average_reasoning_tokens"] for row in rows],
        label=method,
        color=style["color"],
        marker=style["marker"],
        linewidth=3.2,
        markersize=10,
    )

    elevators_label_points.extend(
        {"x": row["floors"], "y": row["average_reasoning_tokens"], "style": style}
        for row in rows
    )

floor_counts = [batch["floors"] for batch in ELEVATORS_BATCHES]
elevators_ax.set_xticks(floor_counts)
elevators_ax.set_xlabel("Number of floors", fontsize=AXIS_LABEL_FONTSIZE, fontweight="semibold", labelpad=12)
elevators_ax.set_ylabel("Average reasoning tokens per problem", fontsize=AXIS_LABEL_FONTSIZE, fontweight="semibold", labelpad=12)
elevators_ax.set_title(
    "DeepSeek Elevators Reasoning Tokens",
    fontsize=TITLE_FONTSIZE,
    fontweight="bold",
    pad=18,
)
elevators_ax.grid(axis="y", color="#D8D1C4", linewidth=0.8, alpha=0.8)
elevators_ax.spines[["top", "right"]].set_visible(False)
elevators_ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
elevators_legend = elevators_ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
elevators_ax.ticklabel_format(axis="y", style="plain")
elevators_ax.margins(y=0.14)
elevators_fig.tight_layout()
annotate_non_overlapping(
    elevators_ax, elevators_label_points, obstacle_artists=[elevators_legend]
)
plt.show()